# Пайплайн обучения, телеметрии и генерации нативной модели D16Qwen с нуля

Этот блокнот реализует полный цикл работы с нативной архитектурой `D16QwenForCausalLM` (`src/d16_qwen_model.py`) на базе объектов `D16ComplexTensor` (`src/d16_complex.py`) без использования внешних весов:
1. **Подготовка токенизатора и обучающей выборки**.
2. **Создание нативной модели `D16QwenForCausalLM`** (каждый нейрон вычисляет $z = \mathrm{D16ComplexTensor.from\_xy}(L, E)$, возводит его в квадрат $z_2 = z.\mathrm{square}()$ внутри клина $[0^\circ, 22.5^\circ]$, модулирует $\mathrm{Re}(z^2)$ и $\mathrm{Im}(z^2)$ 18-секторными таблицами `sector_re_gain` / `sector_im_gain` и проецирует через `down_re_proj` / `down_im_proj`).
3. **Цикл обучения с нуля** с прохождением градиентов через фундаментальный клин $[0^\circ, 22.5^\circ]$ (STE).
4. **Проверка авторегрессионной генерации** и эквивалентности `KV-cache`.
5. **Снятие карты 18 состояний `D16Complex`**, проверка порога состояния `16: Indifferent` (`0xFFFF`) и побитового управления маской сектора (`set_sector_steering`).

In [ ]:
import math
from pathlib import Path
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F

# Подключаем папку src из корня проекта D:\repo\d16nn\qwen\src
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "ipynb" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from d16_complex import D16Complex, D16ComplexTensor, STATE_LABELS
from d16_qwen_model import D16QwenConfig, D16QwenForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
print(f"Using device: {device}")
print(f"Loaded modules from: {SRC_DIR}")

## 1. Подготовка словаря, токенизатора и обучающей выборки

In [ ]:
class SimpleWordTokenizer:
    """Легковесный токенизатор без внешних зависимостей для проверки пайплайна с нуля."""

    def __init__(self, texts):
        special = ["<pad>", "<bos>", "<eos>", "<unk>"]
        words = sorted({tok for text in texts for tok in self._split(text)})
        self.itos = special + words
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.pad_id = 0
        self.bos_id = 1
        self.eos_id = 2
        self.unk_id = 3

    @staticmethod
    def _split(text: str):
        for ch in ["?", ".", ",", ":", ";"]:
            text = text.replace(ch, f" {ch} ")
        return text.strip().split()

    @property
    def vocab_size(self) -> int:
        return len(self.itos)

    def encode(self, text: str, add_bos: bool = True, add_eos: bool = True) -> list[int]:
        ids = [self.stoi.get(t, self.unk_id) for t in self._split(text)]
        if add_bos:
            ids = [self.bos_id] + ids
        if add_eos:
            ids = ids + [self.eos_id]
        return ids

    def decode(self, ids) -> str:
        if isinstance(ids, torch.Tensor):
            ids = ids.tolist()
        tokens = [
            self.itos[i] for i in ids
            if 0 <= i < len(self.itos) and self.itos[i] not in ("<pad>", "<bos>", "<eos>")
        ]
        text = " ".join(tokens)
        for ch in ["?", ".", ",", ":", ";"]:
            text = text.replace(f" {ch}", ch)
        return text


TRAIN_CORPUS = [
    "Q: Does the sun rise in the east? A: Yes, the sun rises in the east and sets in the west.",
    "Q: Is water wet and liquid? A: Yes, water is a clear liquid essential for life.",
    "Q: Is fire cold like ice? A: No, fire is hot and ice is cold.",
    "Q: What is D16Complex? A: D16Complex encodes vectors into the fundamental wedge and a 16-bit sector mask.",
    "Q: How does sector steering work? A: Sector steering rotates only the 16-bit mask without changing wedge coordinates.",
    "Review: The book is inspiring and captivating. Summary: This book is a wonderful masterpiece.",
    "Review: The movie was boring and poorly written. Summary: This movie is a terrible disaster.",
    "Вопрос: Вращается ли Земля вокруг Солнца? Ответ: Да, Земля движется по орбите вокруг Солнца.",
    "Вопрос: Как работает нейрон D16? Ответ: Нейрон вычисляет комплексный квадрат внутри клина двадцать два с половиной градуса.",
]

tokenizer = SimpleWordTokenizer(TRAIN_CORPUS)
encoded_seqs = [torch.tensor(tokenizer.encode(t), dtype=torch.long) for t in TRAIN_CORPUS]
max_len = max(len(s) for s in encoded_seqs)

input_batch = torch.full((len(encoded_seqs), max_len), tokenizer.pad_id, dtype=torch.long)
attn_mask = torch.zeros((len(encoded_seqs), max_len), dtype=torch.long)
for i, seq in enumerate(encoded_seqs):
    input_batch[i, :len(seq)] = seq
    attn_mask[i, :len(seq)] = 1

input_batch = input_batch.to(device)
attn_mask = attn_mask.to(device)

print(f"Размер словаря: {tokenizer.vocab_size} токенов")
print(f"Форма обучающего батча: {tuple(input_batch.shape)}")

## 2. Инициализация нативной модели `D16QwenForCausalLM`

In [ ]:
config = D16QwenConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_size=128,
    intermediate_size=128,
    num_hidden_layers=4,
    num_attention_heads=4,
    num_key_value_heads=2,
    max_position_embeddings=512,
    rms_norm_eps=1e-6,
    rope_theta=10000.0,
    tie_word_embeddings=True,
    indiff_eps=0.0,
)

model = D16QwenForCausalLM(config).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Нативная модель D16QwenForCausalLM создана. Параметров: {total_params:,d}")

## 3. Цикл обучения с нуля

In [ ]:
def train_model(model: D16QwenForCausalLM, steps: int = 120, lr: float = 5e-3):
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=1e-4)

    labels = input_batch.clone()
    labels[input_batch == tokenizer.pad_id] = -100

    history = []
    for step in range(1, steps + 1):
        optimizer.zero_grad()
        out = model(input_ids=input_batch, attention_mask=attn_mask)
        shift_logits = out.logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.view(-1, model.config.vocab_size),
            shift_labels.view(-1),
            ignore_index=-100,
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        loss_val = float(loss.item())
        ppl_val = math.exp(min(loss_val, 20.0))
        history.append((step, loss_val, ppl_val))
        if step == 1 or step % 20 == 0 or step == steps:
            print(f"  Шаг {step:3d}/{steps} | NLL Loss: {loss_val:.4f} | Perplexity: {ppl_val:.2f}")
    return history


print("=== Обучение нативной модели D16Qwen с нуля ===")
history = train_model(model, steps=120, lr=5e-3)

## 4. Проверка генерации и работы `KV-cache`

In [ ]:
test_prompts = [
    "Q: Does the sun rise in the east? A:",
    "Q: Is fire cold like ice? A:",
    "Q: What is D16Complex? A:",
    "Review: The book is inspiring and captivating. Summary:",
    "Вопрос: Как работает нейрон D16? Ответ:",
]

print("=" * 92)
print("ГЕНЕРАЦИЯ ОБУЧЕННОЙ С НУЛЯ НАТИВНОЙ МОДЕЛИ D16Qwen")
print("=" * 92)
model.eval()
for prompt in test_prompts:
    inp = torch.tensor([tokenizer.encode(prompt, add_bos=True, add_eos=False)], dtype=torch.long, device=device)
    out_cached = model.generate(inp, max_new_tokens=16, do_sample=False, eos_token_id=tokenizer.eos_id, use_cache=True)
    out_nocache = model.generate(inp, max_new_tokens=16, do_sample=False, eos_token_id=tokenizer.eos_id, use_cache=False)
    assert torch.equal(out_cached, out_nocache), "Несовпадение между KV-cache и полным проходом!"
    print(f"  Промпт : {prompt}")
    print(f"  Ответ  : {tokenizer.decode(out_cached[0])}")
    print("-" * 92)

## 5. Телеметрия 18 состояний `D16Complex`, порог `0xFFFF` (`Indifferent`) и управление маской сектора

In [ ]:
@torch.no_grad()
def evaluate_corpus_loss(mdl: D16QwenForCausalLM) -> tuple[float, float]:
    mdl.eval()
    labels = input_batch.clone()
    labels[input_batch == tokenizer.pad_id] = -100
    out = mdl(input_ids=input_batch, attention_mask=attn_mask)
    shift_logits = out.logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    loss = F.cross_entropy(
        shift_logits.view(-1, mdl.config.vocab_size),
        shift_labels.view(-1),
        ignore_index=-100,
    )
    nll = float(loss.item())
    return nll, math.exp(min(nll, 20.0))


model.set_indifferent_threshold(0.0)
model.set_collect_d16_stats(True, reset=True)
_ = evaluate_corpus_loss(model)
summary = model.get_d16_state_summary()
counts = summary["state_counts"]
total_el = max(summary["total_elements"], 1)

print("=" * 78)
print("КАРТА 18 СОСТОЯНИЙ D16Complex ПО ВСЕМ СЛОЯМ МОДЕЛИ")
print("=" * 78)
for s_idx in range(18):
    cnt = int(counts[s_idx].item())
    pct = cnt / total_el * 100.0
    print(f"  {s_idx:2d}: {STATE_LABELS[s_idx]:<14} | {cnt:8,d} срабатываний | {pct:6.2f}%")

print("\n" + "=" * 78)
print("ВЛИЯНИЕ ПОРОГА 16:Indifferent (0xFFFF) НА РАЗРЕЖЕННОСТЬ И NLL")
print("=" * 78)
for eps in (0.0, 0.05, 0.15, 0.25, 0.35):
    model.set_indifferent_threshold(eps)
    model.set_collect_d16_stats(True, reset=True)
    nll, ppl = evaluate_corpus_loss(model)
    sp = model.get_d16_state_summary()["indifferent_pct"]
    print(f"  eps = {eps:.2f} | Доля 0xFFFF: {sp:6.2f}% | NLL: {nll:.4f} | PPL: {ppl:.2f}")

model.set_indifferent_threshold(0.0)
model.set_collect_d16_stats(False)

print("\n" + "=" * 78)
print("ПОБИТОВОЕ УПРАВЛЕНИЕ МАСКОЙ СЕКТОРА (КООРДИНАТЫ (x, y) В [0°, 22.5°] НЕИЗМЕННЫ)")
print("=" * 78)
prompt = "Review: The book is inspiring and captivating. Summary:"
inp = torch.tensor([tokenizer.encode(prompt, add_bos=True, add_eos=False)], dtype=torch.long, device=device)

for mode, shift, desc in [
    ("identity", 0, "Исходный сектор (identity)"),
    ("active_swap_le", 0, "Отражение 45° в активных секторах 0..7 (active_swap_le)"),
    ("active_not_logic", 0, "Отражение оси L в активных секторах 0..7 (active_not_logic)"),
    ("rotate", 4, "Поворот маски +90° (rol 4)"),
]:
    model.set_sector_steering(mode=mode, shift=shift, target_layers=[3])
    out_ids = model.generate(inp, max_new_tokens=10, do_sample=False, eos_token_id=tokenizer.eos_id)
    print(f"  {desc:<58} -> {tokenizer.decode(out_ids[0])}")

model.set_sector_steering(mode="identity")

In [ ]:
import importlib
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "ipynb" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import d16_complex
import d16_parameter
import d16_adamw
import d16_qwen_model2
import d16_vis_utils

importlib.reload(d16_complex)
importlib.reload(d16_parameter)
importlib.reload(d16_adamw)
importlib.reload(d16_qwen_model2)
importlib.reload(d16_vis_utils)

from d16_adamw import D16AdamW
from d16_qwen_model2 import D16QwenConfig, D16QwenForCausalLM, encode_token_ids
from d16_vis_utils import (
    plot_d16_sector_transition_matrix,
    plot_d16_weight_sector_distribution,
    print_d16_weight_migration_table,
    snapshot_d16_weight_states,
)

torch.manual_seed(42)
cfg = D16QwenConfig(
    vocab_size=32,
    hidden_size=32,        # 16 комплексных каналов
    intermediate_size=48,  # 48 комплексных каналов в MLP
    num_hidden_layers=2,
    num_attention_heads=4,
    num_key_value_heads=2,
    weight_storage="d16",
)
model_d16 = D16QwenForCausalLM(cfg)
init_snapshot = snapshot_d16_weight_states(model_d16)

optimizer = D16AdamW(model_d16.d16_parameters(), lr=2.5e-2, weight_decay=1e-3)
raw_tokens = torch.tensor(
    [
        [1, 5, 9, 13, 17, 21, 25, 29, 2, 6, 10, 14],
        [3, 7, 11, 15, 19, 23, 27, 31, 4, 8, 12, 16],
    ],
    dtype=torch.long,
)
d16_tokens = encode_token_ids(raw_tokens)

for step in range(30):
    optimizer.zero_grad()
    out = model_d16(input_ids=d16_tokens, labels=d16_tokens)
    out.loss.x.backward()
    optimizer.step(max_grad_norm=1.0)

# 1. Сводная таблица миграции весов D16Parameter
_ = print_d16_weight_migration_table(model_d16, init_snapshot)

# 2. Полярная 16-секторная роза весов и их положение в клине [0°, 22.5°]
fig_dist = plot_d16_weight_sector_distribution(model_d16, init_snapshot)
plt.show()

# 3. Матрица переходов 16x16 между секторами и подвижность по слоям
fig_trans = plot_d16_sector_transition_matrix(model_d16, init_snapshot)
plt.show()

In [ ]:
import importlib
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "ipynb" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import d16_complex
import d16_parameter
import d16_adamw
import d16_qwen_model2
import d16_vis_utils

importlib.reload(d16_complex)
importlib.reload(d16_parameter)
importlib.reload(d16_adamw)
importlib.reload(d16_qwen_model2)
importlib.reload(d16_vis_utils)

from d16_adamw import D16AdamW
from d16_qwen_model2 import D16QwenConfig, D16QwenForCausalLM, encode_token_ids
from d16_vis_utils import (
    D16TrajectoryRecorder,
    plot_d16_sector_trajectories,
    plot_d16_sector_transition_matrix,
    plot_d16_synapse_polar_trajectories,
    plot_d16_weight_sector_distribution,
    print_d16_trajectory_summary,
    print_d16_weight_migration_table,
)

torch.manual_seed(42)
cfg = D16QwenConfig(
    vocab_size=32,
    hidden_size=32,        # 16 комплексных каналов
    intermediate_size=48,  # 48 комплексных каналов в MLP
    num_hidden_layers=2,
    num_attention_heads=4,
    num_key_value_heads=2,
    weight_storage="d16",
)
model_d16 = D16QwenForCausalLM(cfg)

# Инициализируем рекордер траекторий до первого шага оптимизатора
recorder = D16TrajectoryRecorder(model_d16, tracked_synapses_per_group=4)

optimizer = D16AdamW(model_d16.d16_parameters(), lr=2.5e-2, weight_decay=1e-3)
raw_tokens = torch.tensor(
    [
        [1, 5, 9, 13, 17, 21, 25, 29, 2, 6, 10, 14],
        [3, 7, 11, 15, 19, 23, 27, 31, 4, 8, 12, 16],
    ],
    dtype=torch.long,
)
d16_tokens = encode_token_ids(raw_tokens)

for step in range(1, 31):
    optimizer.zero_grad()
    out = model_d16(input_ids=d16_tokens, labels=d16_tokens)
    loss_val = float(out.loss.x.item())
    out.loss.x.backward()
    optimizer.step(max_grad_norm=1.0)
    recorder.record_step(step, model_d16, loss=loss_val)

# 1. Пошаговая таблица траекторий (Attention vs MLP vs Embedding)
print_d16_trajectory_summary(recorder, every_n=5)

# 2. Итоговая таблица по модулям
_ = print_d16_weight_migration_table(model_d16, recorder.initial_snapshot)

# 3. 4-панельный график эволюции секторов и радиусов по шагам
fig_traj = plot_d16_sector_trajectories(recorder)
plt.show()

# 4. Полярные фазовые траектории отдельных синапсов Attention и MLP
fig_syn = plot_d16_synapse_polar_trajectories(recorder)
plt.show()

# 5. Распределение весов в клине [0°, 22.5°] и матрица переходов 16x16
fig_dist = plot_d16_weight_sector_distribution(model_d16, recorder.initial_snapshot)
plt.show()

fig_trans = plot_d16_sector_transition_matrix(model_d16, recorder.initial_snapshot)
plt.show()

In [ ]:
import math
import torch
from d16_adamw import D16AdamW
from d16_qwen_model2 import D16QwenConfig, D16QwenForCausalLM, encode_token_ids
from d16_vis_utils import _iter_complex_d16_parameters, print_d16_weight_migration_table, snapshot_d16_weight_states

raw_tokens = torch.tensor(
    [
        [1, 5, 9, 13, 17, 21, 25, 29, 2, 6, 10, 14],
        [3, 7, 11, 15, 19, 23, 27, 31, 4, 8, 12, 16],
    ],
    dtype=torch.long,
)
d16_tokens = encode_token_ids(raw_tokens)

print("=" * 98)
print(f"{'weight_eps':>10} | {'act_eps':>7} | {'Final Loss':>10} | {'Final PPL':>9} | {'Веса 0xFFFF (%)':>16} | {'Акт. 0xFFFF (%)':>16} | {'Точная ген.':>11}")
print("-" * 98)

best_model = None
best_init_snap = None

for w_eps, a_eps in [(0.0, 0.0), (0.02, 0.10), (0.04, 0.15), (0.06, 0.20), (0.08, 0.25)]:
    torch.manual_seed(42)
    cfg = D16QwenConfig(
        vocab_size=32,
        hidden_size=32,
        intermediate_size=48,
        num_hidden_layers=2,
        num_attention_heads=4,
        num_key_value_heads=2,
        weight_storage="d16",
        indiff_eps=a_eps,
        collect_d16_stats=True,
    )
    model = D16QwenForCausalLM(cfg)
    init_snap = snapshot_d16_weight_states(model)
    optimizer = D16AdamW(model.d16_parameters(), lr=2.5e-2, weight_decay=1e-3)

    for step in range(1, 31):
        optimizer.zero_grad()
        model.set_collect_d16_stats(True, reset=True)
        out = model(input_ids=d16_tokens, labels=d16_tokens)
        out.loss.x.backward()
        optimizer.step(max_grad_norm=1.0)

        # Перевод слабых весов в состояние покоя 16: Indifferent (0xFFFF)
        if w_eps > 0.0:
            with torch.no_grad():
                for _, mod in _iter_complex_d16_parameters(model):
                    mod.copy_(mod.value().apply_Manifold_Threshold_L1(w_eps))

    model.eval()
    with torch.no_grad():
        model.set_collect_d16_stats(True, reset=True)
        out_eval = model(input_ids=d16_tokens, labels=d16_tokens)
        final_loss = float(out_eval.loss.x.item())
        act_sp = model.get_d16_state_summary()["indifferent_pct"]

        tot_w = sum(m.value().numel() for _, m in _iter_complex_d16_parameters(model))
        zero_w = sum(int(m.value().is_zero().sum().item()) for _, m in _iter_complex_d16_parameters(model))
        w_sp = 100.0 * zero_w / max(tot_w, 1)

        gen = model.generate(d16_tokens[:, :4], max_new_tokens=8, do_sample=False, use_cache=True)
        exact = bool((gen.x.to(torch.long) == raw_tokens).all().item())
        ppl = math.exp(min(final_loss, 20.0))

    print(f"{w_eps:10.2f} | {a_eps:7.2f} | {final_loss:10.4f} | {ppl:9.2f} | {w_sp:15.2f}% | {act_sp:15.2f}% | {str(exact):>11}")
    if w_eps == 0.04:
        best_model = model
        best_init_snap = init_snap

print("=" * 98)
# Послойная карта весов для оптимального режима (w_eps=0.04, 20% весов в 0xFFFF, PPL=1.01)
if best_model is not None:
    _ = print_d16_weight_migration_table(best_model, best_init_snap)

In [ ]:
import importlib
import math
import torch

import d16_complex
import d16_parameter
import d16_adamw
import d16_qwen_model
import d16_qwen_model2
import d16_vis_utils

importlib.reload(d16_complex)
importlib.reload(d16_parameter)
importlib.reload(d16_adamw)
importlib.reload(d16_qwen_model)
importlib.reload(d16_qwen_model2)
importlib.reload(d16_vis_utils)

from d16_adamw import D16AdamW
from d16_qwen_model2 import D16QwenConfig, D16QwenForCausalLM, encode_token_ids
from d16_vis_utils import print_d16_weight_migration_table, snapshot_d16_weight_states

torch.manual_seed(42)
cfg = D16QwenConfig(
    vocab_size=32,
    hidden_size=32,
    intermediate_size=48,
    num_hidden_layers=2,
    num_attention_heads=4,
    num_key_value_heads=2,
    weight_storage="d16",
    indiff_eps="auto",   # Автоматический порог 0xFFFF для активаций MLP
    indiff_rel=0.30,
    collect_d16_stats=True,
)
model = D16QwenForCausalLM(cfg)
init_snap = snapshot_d16_weight_states(model)

optimizer = D16AdamW(
    model.d16_parameters(),
    lr=2.5e-2,
    weight_decay=1e-3,
    indiff_eps="auto",   # Автоматический порог 0xFFFF для комплексных весов D16Parameter
    indiff_rel=0.35,
)

raw_tokens = torch.tensor(
    [
        [1, 5, 9, 13, 17, 21, 25, 29, 2, 6, 10, 14],
        [3, 7, 11, 15, 19, 23, 27, 31, 4, 8, 12, 16],
    ],
    dtype=torch.long,
)
d16_tokens = encode_token_ids(raw_tokens)

for step in range(1, 31):
    optimizer.zero_grad()
    model.set_collect_d16_stats(True, reset=True)
    out = model(input_ids=d16_tokens, labels=d16_tokens)
    out.loss.x.backward()
    optimizer.step(max_grad_norm=1.0)

model.eval()
with torch.no_grad():
    model.set_collect_d16_stats(True, reset=True)
    out_eval = model(input_ids=d16_tokens, labels=d16_tokens)
    loss_val = float(out_eval.loss.x.item())
    act_sp = model.get_d16_state_summary()["indifferent_pct"]
    print(f"Final Loss: {loss_val:.4f} | PPL: {math.exp(loss_val):.2f} | Активации в 0xFFFF: {act_sp:.2f}%")

_ = print_d16_weight_migration_table(model, init_snap)

In [ ]:
import importlib
import matplotlib.pyplot as plt
import torch
import d16_vis_utils

importlib.reload(d16_vis_utils)
from d16_vis_utils import plot_d16_attention_heatmap

# Строим тепловую карту для обученной модели с автоматическим порогом indiff_eps="auto"
fig = plot_d16_attention_heatmap(model, d16_tokens, layer_idx=0, head_idx=0)
plt.show()